# Tutorial 2: Analyzing Power Quality Data with DuckDB

This tutorial shows how to use [DuckDB](https://duckdb.org/) to run SQL queries
directly against parquet files on the gateway filesystem. DuckDB reads parquet
natively, so there is no loading step and no extra memory for small queries.

**What you will learn:**
1. Connect DuckDB to local parquet files
2. Query and aggregate PMon data with SQL
3. Query across multiple files with glob patterns
4. Query high-rate CPOW waveform data
5. Combine PMon and CPOW data
6. Convert results to numpy/pandas for plotting

**Why DuckDB + parquet?** Parquet files are columnar and compressed. DuckDB
pushes predicates and projections down into the parquet reader, so a query like
"average voltage between 14:00 and 15:00" only reads the columns and row groups
it needs. For aggregations, filtering, and joins this is more ergonomic than
loading entire tables through pyarrow.

**Prerequisites:** `pip install equser[jupyter]` (includes duckdb, matplotlib, numpy).

## 1. Setup

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.dates import DateFormatter

%matplotlib inline

# Data directory — adjust if running outside the gateway
datadir = Path('/var/lib/eq-coherence/data')
pmon_dir = datadir / 'pmon'
cpow_dir = datadir / 'cpow'

## 2. Connect to parquet files

DuckDB's `read_parquet()` function queries parquet files in place. No data is
copied into memory until you fetch the results. We create an in-memory database
and register views so that later queries can use friendly table names.

In [ ]:
con = duckdb.connect()  # In-memory database

# Pick the most recent PMon file to start with
pmon_files = sorted(pmon_dir.glob('*.parquet'))
if not pmon_files:
    raise FileNotFoundError(f"No PMon parquet files found in {pmon_dir}")

pmon_file = pmon_files[-1]
print(f"Using PMon file: {pmon_file.name}")

In [ ]:
# Create a view so we can query by name
con.execute(f"CREATE OR REPLACE VIEW pmon AS SELECT * FROM read_parquet('{pmon_file}')")

# Inspect the schema
con.sql("DESCRIBE pmon").show()

In [ ]:
# Preview the first few rows
con.sql("SELECT * FROM pmon LIMIT 5").show()

## 3. Query PMon data

PMon files contain one row per measurement interval (~200 ms). Columns include
RMS voltage (AVRMS, BVRMS, CVRMS), RMS current (AIRMS, BIRMS, CIRMS), power
(AWATT, BWATT, CWATT), and frequency (FREQ). See Tutorial 1 for the full column
list.

In [ ]:
# Basic statistics across all three phases
con.sql("""
    SELECT
        MIN(AVRMS) AS va_min, AVG(AVRMS) AS va_avg, MAX(AVRMS) AS va_max,
        MIN(BVRMS) AS vb_min, AVG(BVRMS) AS vb_avg, MAX(BVRMS) AS vb_max,
        MIN(CVRMS) AS vc_min, AVG(CVRMS) AS vc_avg, MAX(CVRMS) AS vc_max
    FROM pmon
""").show()

In [ ]:
# Frequency statistics
con.sql("""
    SELECT
        MIN(FREQ)    AS freq_min,
        AVG(FREQ)    AS freq_avg,
        MAX(FREQ)    AS freq_max,
        STDDEV(FREQ) AS freq_std
    FROM pmon
""").show()

In [ ]:
# Time-range filter: select a 10-minute window
# Adjust the timestamps to match your data.
con.sql("""
    SELECT time_us, AVRMS, BVRMS, CVRMS, FREQ
    FROM pmon
    ORDER BY time_us
    LIMIT 10
""").show()

In [ ]:
# Group by minute to see trends at a coarser resolution
con.sql("""
    SELECT
        date_trunc('minute', time_us) AS minute,
        AVG(AVRMS) AS va_avg,
        AVG(BVRMS) AS vb_avg,
        AVG(CVRMS) AS vc_avg,
        AVG(FREQ)  AS freq_avg,
        COUNT(*)   AS samples
    FROM pmon
    GROUP BY minute
    ORDER BY minute
""").show()

## 4. Multi-file queries

DuckDB can query across many parquet files at once using glob patterns.
This is equivalent to concatenating tables in pyarrow (as in Tutorial 1)
but lets the engine handle the I/O and push down filters.

In [ ]:
# Query across all PMon files
pmon_glob = str(pmon_dir / '*.parquet')
result = con.sql(f"""
    SELECT
        COUNT(*)     AS total_rows,
        MIN(time_us) AS earliest,
        MAX(time_us) AS latest
    FROM read_parquet('{pmon_glob}')
""")
result.show()

In [ ]:
# Hourly averages across all files
con.sql(f"""
    SELECT
        date_trunc('hour', time_us) AS hour,
        AVG(AVRMS) AS va_avg,
        AVG(BVRMS) AS vb_avg,
        AVG(CVRMS) AS vc_avg,
        AVG(AWATT + BWATT + CWATT) AS total_watts_avg,
        COUNT(*) AS samples
    FROM read_parquet('{pmon_glob}')
    GROUP BY hour
    ORDER BY hour
""").show()

In [ ]:
# Find voltage sag candidates: rows where any phase drops below a threshold
# Adjust the threshold (e.g. 114 V for a nominal 120 V system = 95% of nominal).
SAG_THRESHOLD = 114.0

con.sql(f"""
    SELECT time_us, AVRMS, BVRMS, CVRMS
    FROM read_parquet('{pmon_glob}')
    WHERE AVRMS < {SAG_THRESHOLD}
       OR BVRMS < {SAG_THRESHOLD}
       OR CVRMS < {SAG_THRESHOLD}
    ORDER BY time_us
    LIMIT 20
""").show()

## 5. Query CPOW waveform data

CPOW files contain raw 32 kHz waveform samples (VA, VB, VC, IA, IB, IC, IN).
The values are stored as int32 ADC counts; multiply by `vscale` / `iscale` from
the parquet metadata to get volts and amps.

> **Note:** A single 60-second CPOW file has ~1.92 million rows. Use `LIMIT` or
> row-number filtering to avoid pulling more data than you need.

In [ ]:
# Pick the most recent CPOW file
cpow_files = sorted(cpow_dir.glob('*.parquet'))
if not cpow_files:
    raise FileNotFoundError(f"No CPOW parquet files found in {cpow_dir}")

cpow_file = cpow_files[-1]
print(f"Using CPOW file: {cpow_file.name}")

In [ ]:
# Read the scaling factors from parquet metadata
import pyarrow.parquet as pq

pf = pq.ParquetFile(cpow_file)
meta = pf.metadata.metadata or {}
vscale = float(meta[b'vscale'].decode()) if b'vscale' in meta else 1.0
iscale = float(meta[b'iscale'].decode()) if b'iscale' in meta else 1.0
print(f"Voltage scale: {vscale}, Current scale: {iscale}")

In [ ]:
# Query the first 100 ms of waveform data (3,200 samples at 32 kHz)
# and apply the voltage scaling factor in SQL.
cpow_df = con.sql(f"""
    SELECT
        row_number() OVER () AS sample_num,
        VA * {vscale} AS VA,
        VB * {vscale} AS VB,
        VC * {vscale} AS VC,
        IA * {iscale} AS IA,
        IB * {iscale} AS IB,
        IC * {iscale} AS IC
    FROM read_parquet('{cpow_file}')
    LIMIT 3200
""").fetchdf()

print(f"Fetched {len(cpow_df)} samples")
cpow_df.head()

In [ ]:
# Waveform statistics via SQL (no pandas needed)
con.sql(f"""
    SELECT
        MIN(VA * {vscale}) AS va_min,
        MAX(VA * {vscale}) AS va_max,
        MIN(VB * {vscale}) AS vb_min,
        MAX(VB * {vscale}) AS vb_max,
        MIN(VC * {vscale}) AS vc_min,
        MAX(VC * {vscale}) AS vc_max
    FROM read_parquet('{cpow_file}')
""").show()

## 6. Combining PMon and CPOW data

A common workflow is to find an anomaly in PMon summary data and then pull the
corresponding CPOW waveform for closer inspection. Here we use DuckDB to
cross-reference the two data sets.

> **Note:** PMon and CPOW files use different naming conventions. PMon files are
> `YYYYMMDD_HHMM.parquet` (hourly), while CPOW files are
> `YYYYMMDD_HHMMSS.parquet` (60-second). The `filename` column from
> `read_parquet` helps identify source files.

In [ ]:
# Find the minimum voltage across all PMon data (potential sag event)
sag = con.sql(f"""
    SELECT time_us, AVRMS, BVRMS, CVRMS,
           LEAST(AVRMS, BVRMS, CVRMS) AS min_vrms
    FROM read_parquet('{pmon_glob}')
    ORDER BY min_vrms ASC
    LIMIT 1
""").fetchdf()

if len(sag) > 0:
    print(f"Lowest voltage reading at {sag['time_us'].iloc[0]}:")
    print(f"  VA={sag['AVRMS'].iloc[0]:.2f} V, "
          f"VB={sag['BVRMS'].iloc[0]:.2f} V, "
          f"VC={sag['CVRMS'].iloc[0]:.2f} V")

In [ ]:
# List CPOW files with their time coverage
# The filename encodes the start time as YYYYMMDD_HHMMSS
cpow_glob = str(cpow_dir / '*.parquet')
con.sql(f"""
    SELECT
        filename,
        COUNT(*) AS samples,
        COUNT(*) / 32000.0 AS duration_sec
    FROM read_parquet('{cpow_glob}', filename=true)
    GROUP BY filename
    ORDER BY filename
""").show()

## 7. Results to plots

DuckDB results convert to pandas DataFrames with `.fetchdf()` or to numpy arrays
with `.fetchnumpy()`. Both integrate directly with matplotlib.

In [ ]:
# Fetch minute-averaged voltage from all PMon files
trend_df = con.sql(f"""
    SELECT
        date_trunc('minute', time_us) AS minute,
        AVG(AVRMS) AS va_avg,
        AVG(BVRMS) AS vb_avg,
        AVG(CVRMS) AS vc_avg
    FROM read_parquet('{pmon_glob}')
    GROUP BY minute
    ORDER BY minute
""").fetchdf()

fig, ax = plt.subplots()
ax.set_title('RMS Voltage (minute averages)')
ax.plot(trend_df['minute'], trend_df['va_avg'], 'k', label='VA', zorder=3, alpha=0.7)
ax.plot(trend_df['minute'], trend_df['vb_avg'], 'r', label='VB', zorder=2, alpha=0.7)
ax.plot(trend_df['minute'], trend_df['vc_avg'], 'b', label='VC', zorder=1, alpha=0.7)
ax.set_ylabel('RMS Voltage (V)')
ax.set_xlabel('Time (UTC)')
ax.xaxis.set_major_formatter(DateFormatter('%H:%M'))
ax.legend()
plt.show()
plt.close()

In [ ]:
# Plot the CPOW waveform slice we queried earlier
SAMPLE_RATE_HZ = 32_000
time_ms = np.arange(len(cpow_df)) / SAMPLE_RATE_HZ * 1000

fig, ax = plt.subplots()
ax.set_title('Voltage Waveforms')
ax.set_xlabel('Elapsed time [ms]')
ax.set_ylabel('Voltage [V]')
ax.plot(time_ms, cpow_df['VA'].values, 'k', label='VA', zorder=3, alpha=0.7)
ax.plot(time_ms, cpow_df['VB'].values, 'r', label='VB', zorder=2, alpha=0.7)
ax.plot(time_ms, cpow_df['VC'].values, 'b', label='VC', zorder=1, alpha=0.7)
ax.legend(bbox_to_anchor=(1.0, 0.5))
plt.show()
plt.close()

In [ ]:
# Frequency trend
freq_df = con.sql(f"""
    SELECT time_us, FREQ
    FROM read_parquet('{pmon_glob}')
    ORDER BY time_us
""").fetchdf()

fig, ax = plt.subplots()
ax.set_title('Frequency')
ax.plot(freq_df['time_us'], freq_df['FREQ'], 'k', linewidth=0.5)
ax.set_ylabel('Frequency [Hz]')
ax.set_xlabel('Time (UTC)')
ax.xaxis.set_major_formatter(DateFormatter('%H:%M'))
plt.show()
plt.close()

## 8. Comparison with pyarrow

Both DuckDB and pyarrow can read parquet files. Here is a quick guide on when
to reach for each one.

| Task | DuckDB (SQL) | pyarrow (programmatic) |
|------|-------------|------------------------|
| Aggregations (AVG, MIN, MAX, GROUP BY) | Natural fit | Requires `compute` calls or pandas |
| Time-range filtering | `WHERE time_us BETWEEN ...` | Row-group pruning with filters= |
| Multi-file queries | `read_parquet('*.parquet')` | `pq.ParquetDataset` or manual concat |
| Joins across data types | SQL `JOIN` | Manual merge logic |
| Sample-level waveform slicing | `LIMIT` / `OFFSET` or `row_number()` | Python slicing, more flexible |
| Streaming / incremental reads | Not ideal | `iter_batches()` for memory control |
| Custom numpy operations | Fetch then process | Direct array access |

**Rule of thumb:** Use DuckDB when your question is naturally expressed as SQL
(aggregations, filters, joins, grouped statistics). Use pyarrow or numpy when
you need fine-grained array manipulation, streaming reads, or direct integration
with signal processing code.

In [ ]:
# Side-by-side example: average voltage per phase
import pyarrow.compute as pc
import pyarrow.parquet as pq

# --- DuckDB approach ---
duckdb_result = con.sql(f"""
    SELECT AVG(AVRMS) AS va, AVG(BVRMS) AS vb, AVG(CVRMS) AS vc
    FROM read_parquet('{pmon_file}')
""").fetchdf()
print("DuckDB result:")
print(duckdb_result)

# --- pyarrow approach ---
table = pq.read_table(pmon_file)
va_avg = pc.mean(table['AVRMS']).as_py()
vb_avg = pc.mean(table['BVRMS']).as_py()
vc_avg = pc.mean(table['CVRMS']).as_py()
print(f"\npyarrow result: VA={va_avg:.4f}, VB={vb_avg:.4f}, VC={vc_avg:.4f}")

## Next steps

- **Tutorial 1** (`01-parquet-files.ipynb`): Load and plot data directly with pyarrow and `equser` plotters.
- **Tutorial 3** (`03-backend-api.ipynb`): Query data through the REST API (also backed by DuckDB on the server).
- **Tutorial 4** (`04-live-streaming.ipynb`): Connect to live WebSocket streams for real-time data.

In [ ]:
# Clean up
con.close()